# PrometheusNet training on PUMA

Colab Pro entry point for the instance-aware multitask architecture. Tissue is trained as semantic segmentation; nuclei are trained as center-based instances and evaluated with centroid F1.

## 1. Environment

In [ ]:
import os, sys
REPO_DIR = '/content/prometheus'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Prometheus.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -qq -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f'PyTorch {torch.__version__}; CUDA {torch.version.cuda}')
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, torch.cuda.get_device_name(index), f'{props.total_memory / 2**30:.1f} GiB')


## 2. Resolved configuration

In [ ]:
from dataclasses import asdict
from pprint import pprint
from prometheus.api import load_config, build_datamodule, build_model, build_trainer

CONFIG_PATH = 'configs/experiment/baseline_multitask.toml'
config = load_config(CONFIG_PATH)
config.data.root = '/content/drive/MyDrive/PUMA/dataset_PUMA'
config.paths.run_dir = '/content/drive/MyDrive/prometheus_runs/baseline_multitask_v1'
# Reduce batch size and increase accumulation if Colab runs out of VRAM.
config.trainer.batch_size = 4
config.trainer.gradient_accumulation = 1
config.validate()
pprint(asdict(config))


## 3. Data and one-batch smoke test

In [ ]:
train_loader, validation_loader = build_datamodule(config)
batch = next(iter(train_loader))
print('images', tuple(batch.images.shape))
print('tissue', tuple(batch.tissue.mask.shape))
print('instances/image', [len(target.labels) for target in batch.nuclei])


In [ ]:
from prometheus.losses import LossWeights, PrometheusMultitaskLoss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(config).to(device)
batch = batch.to(device)
output = model(batch.images)
criterion = PrometheusMultitaskLoss(
    config.model.num_nucleus_types,
    config.model.nuclei_feature_stride,
    LossWeights(**config.loss.__dict__),
)
losses = criterion(output, batch)
losses['total'].backward()
print({name: round(value.detach().item(), 5) for name, value in losses.items()})
model.zero_grad(set_to_none=True)
torch.cuda.empty_cache()


## 4. Train or resume

In [ ]:
from pathlib import Path
model = build_model(config)
trainer = build_trainer(config, model=model, datamodule=(train_loader, validation_loader), device=device)
last_checkpoint = Path(config.paths.run_dir) / 'last.ckpt'
resume_from = last_checkpoint if last_checkpoint.exists() else None
metrics = trainer.fit(resume_from=resume_from)
print(metrics)


## 5. Exact validation and artifacts

In [ ]:
from prometheus.engine import evaluate_multitask
best_path = Path(config.paths.run_dir) / 'best_primary.ckpt'
trainer.resume(best_path)
result = evaluate_multitask(
    trainer.model, validation_loader, trainer.criterion, device,
    config.model.nuclei_feature_stride, config.evaluation.nuclei_radius_px,
    config.postprocess.confidence_threshold, config.postprocess.max_detections,
    config.postprocess.local_max_kernel,
)
print('tissue Dice:', result.tissue_dice)
print('nuclei centroid F1:', result.nuclei_macro_f1)
print('checkpoint:', best_path)


## 6. Submission smoke test

Use the same checkpoint through the CLI so notebook and headless inference share one composition path:

```bash
prometheus predict --config configs/experiment/baseline_multitask.toml --checkpoint /path/to/best_primary.ckpt --input /path/to/sample.tif --output /content/prediction
```